In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


INPUT_ROOT = Path("/kaggle/input")

canonical_matches = list(
    INPUT_ROOT.rglob(
        "router_project_canonical.pkl"
    )
)

if not canonical_matches:
    raise FileNotFoundError(
        "router_project_canonical.pkl not found"
    )

CANONICAL_PATH = canonical_matches[0]

canonical_df = pd.read_pickle(
    CANONICAL_PATH
)

print("Loaded:", CANONICAL_PATH)
print("Shape:", canonical_df.shape)


X = canonical_df[
    [
        "sample_id",
        "prompt",
        "domain",
        "eval_name"
    ]
].copy()


# Reproduce the ORIGINAL split exactly
all_indices = np.arange(
    len(X)
)

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42,
    stratify=X["domain"]
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=X.loc[
        temp_idx,
        "domain"
    ]
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Loaded: /kaggle/input/datasets/paam24/llm-data/router_project_canonical.pkl
Shape: (22962, 26)
Train: 18369
Validation: 2296
Test: 2297


In [2]:

# Keep the main training stack aligned with the original experiment.
# We intentionally do not reinstall PyTorch because Kaggle provides a GPU-ready build.

!pip -q install \
    transformers==5.17.0 \
    datasets==4.8.5 \
    accelerate==1.15.0 \
    scikit-learn==1.6.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 101.7 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 62.4 MB/s eta 0:00:00:00:01


In [3]:

import gc
import shutil
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import sklearn
import datasets
import accelerate

from datasets import Dataset

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected. In Kaggle Notebook Settings, "
        "set Accelerator to GPU before running this notebook."
    )


Python: 3.12.13
Platform: Linux-6.18.48+-x86_64-with-glibc2.35
PyTorch: 2.10.0+cu128
Transformers: 5.17.0
Datasets: 4.8.5
Accelerate: 1.15.0
scikit-learn: 1.6.1
CUDA available: True
GPU: Tesla T4


In [5]:

# Automatically find the canonical dataset anywhere under /kaggle/input.
# The original split is recreated exactly, so router_split_indices.npz is NOT required.

from sklearn.model_selection import train_test_split

INPUT_ROOT = Path("/kaggle/input")

canonical_matches = list(
    INPUT_ROOT.rglob("router_project_canonical.pkl")
)

if not canonical_matches:
    raise FileNotFoundError(
        "router_project_canonical.pkl was not found under /kaggle/input. "
        "Attach/upload it as a Kaggle notebook input."
    )

CANONICAL_PATH = canonical_matches[0]

print("Canonical dataset:", CANONICAL_PATH)

canonical_df = pd.read_pickle(
    CANONICAL_PATH
)

MODEL_COLS = [
    "WizardLM/WizardLM-13B-V1.2",
    "claude-instant-v1",
    "claude-v1",
    "claude-v2",
    "gpt-3.5-turbo-1106",
    "gpt-4-1106-preview",
    "meta/code-llama-instruct-34b-chat",
    "meta/llama-2-70b-chat",
    "mistralai/mistral-7b-chat",
    "mistralai/mixtral-8x7b-chat",
    "zero-one-ai/Yi-34B-Chat"
]

X = canonical_df[
    ["sample_id", "prompt", "domain", "eval_name"]
].copy()

Y = pd.DataFrame({
    model: canonical_df[f"{model}|performance"]
    for model in MODEL_COLS
}).astype("float32")

C = pd.DataFrame({
    model: canonical_df[f"{model}|cost"]
    for model in MODEL_COLS
}).astype("float32")

# Recreate the EXACT original 80/10/10 stratified split.
all_indices = np.arange(len(X))

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42,
    stratify=X["domain"]
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=X.loc[temp_idx, "domain"]
)

X_train = X.iloc[train_idx].reset_index(drop=True)
X_val = X.iloc[val_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)

Y_train = Y.iloc[train_idx].reset_index(drop=True)
Y_val = Y.iloc[val_idx].reset_index(drop=True)
Y_test = Y.iloc[test_idx].reset_index(drop=True)

C_train = C.iloc[train_idx].reset_index(drop=True)
C_val = C.iloc[val_idx].reset_index(drop=True)
C_test = C.iloc[test_idx].reset_index(drop=True)

mean_train_cost = C_train.mean().to_numpy()

GPT4_INDEX = MODEL_COLS.index(
    "gpt-4-1106-preview"
)

print("Canonical shape:", canonical_df.shape)
print("Train / Val / Test:", len(X_train), len(X_val), len(X_test))

assert canonical_df.shape == (22962, 26)
assert len(X_train) == 18369
assert len(X_val) == 2296
assert len(X_test) == 2297


Canonical dataset: /kaggle/input/datasets/paam24/llm-data/router_project_canonical.pkl
Canonical shape: (22962, 26)
Train / Val / Test: 18369 2296 2297


In [6]:

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 384

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

def make_dataset(X_part, Y_part):
    return Dataset.from_dict({
        "text": X_part["prompt"].tolist(),
        "labels": Y_part.to_numpy(
            dtype=np.float32
        ).tolist()
    })


train_ds = make_dataset(X_train, Y_train)
val_ds = make_dataset(X_val, Y_val)
test_ds = make_dataset(X_test, Y_test)


def tokenize_batch(batch):
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

    # DistilBERT does not use token_type_ids
    encoded.pop("token_type_ids", None)

    return encoded


train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

val_tok = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

test_tok = test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print(train_tok, val_tok, test_tok)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/18369 [00:00<?, ? examples/s]

Map:   0%|          | 0/2296 [00:00<?, ? examples/s]

Map:   0%|          | 0/2297 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 18369
}) Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 2296
}) Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 2297
})


In [7]:

def regression_metrics(eval_pred):
    pred = np.clip(
        eval_pred.predictions,
        0.0,
        1.0
    )

    labels = eval_pred.label_ids

    return {
        "mse": mean_squared_error(
            labels,
            pred
        ),
        "mae": mean_absolute_error(
            labels,
            pred
        )
    }


def route_predictions(predictions, threshold):
    cost_order = np.argsort(
        mean_train_cost
    )

    choices = []

    for scores in predictions:
        eligible = [
            i for i in cost_order
            if scores[i] >= threshold
        ]

        if eligible:
            selected = eligible[0]
        else:
            selected = int(
                np.argmax(scores)
            )

        choices.append(selected)

    return np.asarray(
        choices,
        dtype=int
    )


def realised_metrics(predictions, y_np, c_np, threshold):
    choices = route_predictions(
        predictions,
        threshold
    )

    rows = np.arange(
        len(choices)
    )

    perf = y_np[
        rows,
        choices
    ]

    cost = c_np[
        rows,
        choices
    ]

    return choices, perf, cost


In [8]:

Y_val_np = Y_val.to_numpy()
Y_test_np = Y_test.to_numpy()
C_val_np = C_val.to_numpy()
C_test_np = C_test.to_numpy()

gpt4_val_perf = (
    Y_val_np[:, GPT4_INDEX].mean()
)

gpt4_test_perf = (
    Y_test_np[:, GPT4_INDEX].mean()
)

gpt4_test_cost = (
    C_test_np[:, GPT4_INDEX].mean()
)

TARGET_VAL_PERF = (
    gpt4_val_perf - 0.01
)

FINE_THRESHOLDS = np.round(
    np.arange(0.90, 1.001, 0.01),
    2
)

print("Validation target:", TARGET_VAL_PERF)


Validation target: 0.76504356


In [9]:

def choose_validation_threshold(val_pred):
    rows = []

    for theta in FINE_THRESHOLDS:
        _, perf, cost = realised_metrics(
            val_pred,
            Y_val_np,
            C_val_np,
            theta
        )

        rows.append({
            "threshold": float(theta),
            "performance": perf.mean(),
            "cost": cost.mean()
        })

    curve = pd.DataFrame(rows)

    acceptable = curve[
        curve["performance"] >= TARGET_VAL_PERF
    ]

    if len(acceptable):
        point = acceptable.loc[
            acceptable["cost"].idxmin()
        ]
    else:
        point = curve.loc[
            curve["performance"].idxmax()
        ]

    return float(point["threshold"]), curve


In [10]:

SEEDS = [13, 42, 77]

seed_results = []

for seed in SEEDS:

    print("\n" + "=" * 70)
    print("SEED:", seed)
    print("=" * 70)

    set_seed(seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(MODEL_COLS),
        problem_type="regression"
    )

    output_dir = (
        f"/kaggle/working/distilbert_multiseed_{seed}"
    )

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        weight_decay=0.01,
        warmup_steps=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="mae",
        greater_is_better=False,
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        save_total_limit=1,
        seed=seed,
        data_seed=seed,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=collator,
        compute_metrics=regression_metrics,
        processing_class=tokenizer
    )

    trainer.train()

    val_output = trainer.predict(
        val_tok
    )

    test_output = trainer.predict(
        test_tok
    )

    val_pred = np.clip(
        val_output.predictions,
        0.0,
        1.0
    )

    test_pred = np.clip(
        test_output.predictions,
        0.0,
        1.0
    )

    threshold, curve = (
        choose_validation_threshold(
            val_pred
        )
    )

    test_choices, test_perf, test_cost = (
        realised_metrics(
            test_pred,
            Y_test_np,
            C_test_np,
            threshold
        )
    )

    result = {
        "seed": seed,
        "threshold": threshold,
        "val_mse": mean_squared_error(
            Y_val_np,
            val_pred
        ),
        "val_mae": mean_absolute_error(
            Y_val_np,
            val_pred
        ),
        "test_mse": mean_squared_error(
            Y_test_np,
            test_pred
        ),
        "test_mae": mean_absolute_error(
            Y_test_np,
            test_pred
        ),
        "test_router_performance":
            test_perf.mean(),
        "test_router_cost":
            test_cost.mean(),
        "performance_gap_pp":
            (
                gpt4_test_perf
                - test_perf.mean()
            ) * 100,
        "cost_saving_percent":
            (
                1
                - test_cost.mean()
                / gpt4_test_cost
            ) * 100
    }

    seed_results.append(result)

    print("\nResult:")
    print(result)

    curve.to_csv(
        f"/kaggle/working/multiseed_val_curve_seed_{seed}.csv",
        index=False
    )

    del trainer
    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    shutil.rmtree(
        output_dir,
        ignore_errors=True
    )



SEED: 13


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super(

Epoch,Training Loss,Validation Loss,Mse,Mae
1,0.260915,0.255789,0.127885,0.270365
2,0.247696,0.244103,0.121998,0.269969
3,0.235775,0.243994,0.121928,0.264657


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Result:
{'seed': 77, 'threshold': 0.92, 'val_mse': 0.12337059527635574, 'val_mae': 0.26544389128685, 'test_mse': 0.12434690445661545, 'test_mae': 0.26645997166633606, 'test_router_performance': np.float32(0.76033956), 'test_router_cost': np.float32(0.0029277557), 'performance_gap_pp': np.float32(0.7727504), 'cost_saving_percent': np.float32(21.681559)}


In [11]:

seed_results_df = pd.DataFrame(
    seed_results
)

display(seed_results_df)

seed_results_df.to_csv(
    "/kaggle/working/distilbert_multiseed_results.csv",
    index=False
)


,seed,threshold,val_mse,val_mae,test_mse,test_mae,test_router_performance,test_router_cost,performance_gap_pp,cost_saving_percent
0,13,0.92,0.121928,0.264657,0.122949,0.265853,0.760993,0.003072,0.707448,17.818462
1,42,0.94,0.122822,0.263210,0.123765,0.264452,0.762952,0.002899,0.511539,22.460943
2,77,0.92,0.123371,0.265444,0.124347,0.266460,0.760340,0.002928,0.772750,21.681559


In [12]:

metrics_to_summarise = [
    "threshold",
    "val_mse",
    "val_mae",
    "test_mse",
    "test_mae",
    "test_router_performance",
    "test_router_cost",
    "performance_gap_pp",
    "cost_saving_percent"
]

summary_rows = []

for metric in metrics_to_summarise:
    summary_rows.append({
        "metric": metric,
        "mean": seed_results_df[
            metric
        ].mean(),
        "sd": seed_results_df[
            metric
        ].std(ddof=1)
    })

multiseed_summary = pd.DataFrame(
    summary_rows
)

display(multiseed_summary)

multiseed_summary.to_csv(
    "/kaggle/working/distilbert_multiseed_mean_sd.csv",
    index=False
)


,metric,mean,sd
0,threshold,0.926667,0.011547
1,val_mse,0.122707,0.000728
2,val_mae,0.264437,0.001133
3,test_mse,0.123687,0.000702
4,test_mae,0.265588,0.001030
5,test_router_performance,0.761428,0.001359
6,test_router_cost,0.002966,0.000093
7,performance_gap_pp,0.663912,0.135939
8,cost_saving_percent,20.653654,2.486080


In [13]:

print("REPORT-READY SUMMARY\n")

for _, row in multiseed_summary.iterrows():
    print(
        f'{row["metric"]}: '
        f'{row["mean"]:.6f} ± '
        f'{row["sd"]:.6f}'
    )


REPORT-READY SUMMARY

threshold: 0.926667 ± 0.011547
val_mse: 0.122707 ± 0.000728
val_mae: 0.264437 ± 0.001133
test_mse: 0.123687 ± 0.000702
test_mae: 0.265588 ± 0.001030
test_router_performance: 0.761428 ± 0.001359
test_router_cost: 0.002966 ± 0.000093
performance_gap_pp: 0.663912 ± 0.135939
cost_saving_percent: 20.653654 ± 2.486080


In [14]:

from pathlib import Path

print("Saved Kaggle outputs:")

for path in sorted(
    Path("/kaggle/working").glob("*.csv")
):
    print(
        path.name,
        f"({path.stat().st_size / 1024:.1f} KB)"
    )


Saved Kaggle outputs:
distilbert_multiseed_mean_sd.csv (0.5 KB)
distilbert_multiseed_results.csv (0.5 KB)
multiseed_val_curve_seed_13.csv (0.3 KB)
multiseed_val_curve_seed_42.csv (0.3 KB)
multiseed_val_curve_seed_77.csv (0.3 KB)
